# Haemato-Oncology Cytogenetics: From Free-Text HL7 v2 to Structured Observations

This is the twelfth notebook in the series. `Input/V2/R01/Shire-1.txt` and
`Shire-2.txt` are two real Haemato-Oncology cytogenetics reports - Shire (MFT's
cellular pathology LIMS) reporting through HODS, the Haemato-Oncology order comms
system, per the
[Haemato-Oncology Diagnostic Pathway](https://nw-gmsa.github.io/en/HaematoOncologyPathway.html):
a single referral for suspected blood cancer triggers pathology assessment and a
conditional genomic reflex test, combined into one report back to the referrer.
Today, both reports carry their entire karyotype/FISH findings as free text - "every
line of the report is a separate `OBX|n|FT|CYTO||...` segment", in that page's own
words, with `OBR-4` (the test itself) left uncoded.

- **Step 1** extracts the individual variables (the ISCN karyotype string, the FISH
  result, the conclusion, amendment markers) out of that free text.
- **Step 2** looks at which LOINC codes and panels NW-GMSA's own page proposes for
  representing them as structured `Observation`s, and builds them.

## The pathway, and why the free text is a problem

[nw-gmsa.github.io/en/HaematoOncologyPathway.html](https://nw-gmsa.github.io/en/HaematoOncologyPathway.html):
initial referral → pathology assessment (`LAB-35`/`LAB-36`) → conditional genomic
reflex testing (`LAB-35`/`LAB-36`) → one combined report (`LAB-3`) back to the
referrer. The two fixtures here are that final cytogenetics report, from two
different real patients: `Shire-1.txt` (a single 20q deletion, MDS) and `Shire-2.txt`
(a complex hyperdiploid AML karyotype plus a FISH result, with inline
`-Amendment 14/10/20` markers).

The page's own critique of the current shape: with the whole report as one run of
`OBX|n|FT|CYTO||...` lines, a downstream system gets a block of prose to display, not
data to query, trend, or code against - and amendments show up as inline text rather
than a formal status change. Its proposed fix, which this notebook builds:
`DiagnosticReport.conclusion` for the free-text interpretive summary,
`DiagnosticReport.conclusionCode` for a coded diagnosis/impression, discrete
`Observation`s (coded to LOINC) for the karyotype and FISH findings themselves, and
`DiagnosticReport.status = corrected` with a linked `Provenance` history in place of
the inline amendment markers.

In [1]:
import json
import os
import subprocess
import tempfile
from uuid import uuid4


def read_segments(path):
    with open(path, newline="") as f:
        raw = f.read()
    return [s for s in raw.replace("\r\n", "\r").split("\r") if s]


def by_segment_type(segments):
    grouped = {}
    for segment in segments:
        fields = segment.split("|")
        grouped.setdefault(fields[0], []).append(fields)
    return grouped


NHS_NUMBER_SYSTEM = "https://fhir.nhs.uk/Id/nhs-number"
ODS_SYSTEM = "https://fhir.nhs.uk/Id/ods-organization-code"
V2_0203 = "http://terminology.hl7.org/CodeSystem/v2-0203"
LOINC_SYSTEM = "http://loinc.org"


## Today's baseline: one `Observation`, one giant string

Before building anything, worth seeing exactly what the live RIE already produces for
these two files - `Output/FHIR/R01/Shire-1.txt.json`/`Shire-2.txt.json`, from an
earlier real `transformToFHIR` run (see `Shire.md`). One `Observation`, SNOMED CT
`1054161000000101` "Genetics Report", `Observation.valueString` holding the *entire*
report - literally the raw text with `\r\n`s intact - as a single opaque string. The
one thing already coded is `DiagnosticReport.code` itself:
`33893-9` "Karyotype [Identifier] in Bone marrow Nominal" - a real LOINC code, and
one this notebook's own `DiagnosticReport`s below reuse rather than replace.

In [2]:
for path in ("Output/FHIR/R01/Shire-1.txt.json", "Output/FHIR/R01/Shire-2.txt.json"):
    with open(path) as f:
        bundle = json.load(f)
    observation = next(e["resource"] for e in bundle["entry"] if e["resource"]["resourceType"] == "Observation")
    report = next(e["resource"] for e in bundle["entry"] if e["resource"]["resourceType"] == "DiagnosticReport")
    print(path)
    print(" DiagnosticReport.code:", report["code"]["coding"][0]["code"], report["code"]["coding"][0]["display"])
    print(" Observation.code:     ", observation["code"]["coding"][0]["code"], observation["code"]["coding"][0]["display"])
    print(" Observation.valueString (first 120 chars):", repr(observation["valueString"][:120]))
    print()

Output/FHIR/R01/Shire-1.txt.json
 DiagnosticReport.code: 33893-9 Karyotype [Identifier] in Bone marrow Nominal
 Observation.code:      1054161000000101 Genetics Report
 Observation.valueString (first 120 chars): 'Karyotype: 46,XY,del(20)(q*q*)[]\r\n\r\nChromosome analysis has shown an abnormal karyotype with a deletion in the long arm\r'

Output/FHIR/R01/Shire-2.txt.json
 DiagnosticReport.code: 33893-9 Karyotype [Identifier] in Bone marrow Nominal
 Observation.code:      1054161000000101 Genetics Report
 Observation.valueString (first 120 chars): 'Karyotype:\r\n48/R/50,XY,+X,-Y,-3,-6,-7,del(7)(q?31q36),+8,+11,add(12)(p11.2),+14,+15,-16,+19,+20,+22-\r\n,+1/R/2mar[cp10] -'



## Step 1: extracting the variables from HL7 v2

Same starting point as `04-laboratory-report-fhir-from-hl7v2.ipynb` - split on `\r`,
group by segment type - except here the payload of interest is entirely inside
`OBX-5` (`FT`, free text), one line of the report per `OBX`, in `OBX-1` (Set ID)
order.

In [3]:
def report_text(path):
    segments = read_segments(path)
    by_seg = by_segment_type(segments)
    lines = [obx[5] if len(obx) > 5 else "" for obx in by_seg["OBX"]]
    return "\n".join(lines), by_seg


shire1_text, shire1_by_seg = report_text("Input/V2/R01/Shire-1.txt")
shire2_text, shire2_by_seg = report_text("Input/V2/R01/Shire-2.txt")

print("=== Shire-1.txt ===")
print(shire1_text)
print()
print("=== Shire-2.txt ===")
print(shire2_text)

=== Shire-1.txt ===
Karyotype: 46,XY,del(20)(q*q*)[]

Chromosome analysis has shown an abnormal karyotype with a deletion in the long arm
of chromosome 20.

Deletion of chromosome 20q is a common but non-specific finding in myeloid disorders
and is consistent with the diagnosis of MDS. In particular, according to IPS, deletion
of 20q as the sole abnormality is associated with a good prognosis in MDS, but should
be interpreted in the context of other biological and clinical prognostic factors.

Conclusion: 

=== Shire-2.txt ===
Karyotype:
48\R\50,XY,+X,-Y,-3,-6,-7,del(7)(q?31q36),+8,+11,add(12)(p11.2),+14,+15,-16,+19,+20,+22-
,+1\R\2mar[cp10] -Amendment 14/10/20

FISH Result: 7q deletion detected. -Amendment 14/10/20

-Amendment 14/10/20
Chromosome analysis has shown all 10 cells examined to have abnormal hyperdiploid
karyotype with abnormalities which included: a deletion of the long arm of chromosome
7; additional material on the short arm of chromosome 12 (resulting in loss of 12p);


### An HL7 v2 wrinkle: escaped ISCN

`Shire-2.txt`'s karyotype spans three `OBX` lines, and the middle one ends
`...+22-` while the next starts `,+1\R\2mar[cp10]`. `\R\` is an HL7 v2 escape
sequence - it means "the repetition separator character itself" (`~`, `MSH-2`'s third
encoding character), escaped so it doesn't get parsed as a real field repetition
inside this `FT` value. Decoded, `48\R\50` is really `48~50` - a mosaic/composite
modal chromosome count range, valid ISCN. Skip this decoding step and the "structured"
karyotype string a downstream system stores is subtly wrong; get it right and the
three wrapped `OBX` lines concatenate (no separator - they're one continuous
expression, not three lines of prose) into a clean, valid ISCN string.

Worth checking against the baseline above: the live RIE's own already-committed output
for `Shire-2.txt.json` (the baseline cell, above) shows `48/R/50` - forward slashes,
not the `~` this decodes to. The real engine mangles this exact escape sequence rather
than resolving it; getting the karyotype string right here isn't a hypothetical
concern.

In [4]:
import re


def decode_v2_escapes(s):
    return (s.replace("\\F\\", "|").replace("\\S\\", "^")
             .replace("\\T\\", "&").replace("\\R\\", "~")
             .replace("\\E\\", "\\"))


AMENDMENT_RE = re.compile(r"\s*-Amendment (\d{2}/\d{2}/\d{2})\s*$")


def extract_karyotype(text):
    lines = text.split("\n")
    start = next(i for i, l in enumerate(lines) if l.strip().startswith("Karyotype:"))
    span = []
    first = lines[start].split("Karyotype:", 1)[1].strip()
    if first:
        span.append(first)
    for l in lines[start + 1:]:
        if l.strip() == "":
            break
        span.append(l)
    raw = "".join(span)
    m = AMENDMENT_RE.search(raw)
    amendment = m.group(1) if m else None
    if m:
        raw = raw[: m.start()]
    return decode_v2_escapes(raw), amendment


for label, text in (("Shire-1", shire1_text), ("Shire-2", shire2_text)):
    karyotype, amendment = extract_karyotype(text)
    print(f"{label}: karyotype = {karyotype!r}")
    print(f"{label}: amendment = {amendment!r}")
    print()

Shire-1: karyotype = '46,XY,del(20)(q*q*)[]'
Shire-1: amendment = None

Shire-2: karyotype = '48~50,XY,+X,-Y,-3,-6,-7,del(7)(q?31q36),+8,+11,add(12)(p11.2),+14,+15,-16,+19,+20,+22-,+1~2mar[cp10]'
Shire-2: amendment = '14/10/20'



In [5]:
def extract_conclusion(text):
    m = re.search(r"Conclusion:\s*(.*)\Z", text, re.S)
    if not m:
        return None
    conclusion = m.group(1).strip()
    m2 = AMENDMENT_RE.search(conclusion)
    if m2:
        conclusion = conclusion[: m2.start()].strip()
    return conclusion


def extract_fish(text):
    m = re.search(r"FISH Result:\s*(.*)", text)
    if not m:
        return None
    result = AMENDMENT_RE.sub("", m.group(1)).strip()
    counts = re.search(r"(\d+) out of (\d+) interphase cells", text)
    return {
        "result": result,
        "positive_cells": int(counts.group(1)) if counts else None,
        "cells_examined": int(counts.group(2)) if counts else None,
    }


for label, text in (("Shire-1", shire1_text), ("Shire-2", shire2_text)):
    conclusion = extract_conclusion(text)
    fish = extract_fish(text)
    print(f"{label}: conclusion = {conclusion!r}")
    print(f"{label}: fish       = {fish!r}")
    print()

Shire-1: conclusion = ''
Shire-1: fish       = None

Shire-2: conclusion = 'Complex abnormal hyperdiploid karyotype with  gains of chromosomes 8 and\n11, and loss of 7q and 12p which  are recognised findings in myeloid neoplasms\nincluding AML'
Shire-2: fish       = {'result': '7q deletion detected.', 'positive_cells': 93, 'cells_examined': 100}



`Shire-1.txt`'s `Conclusion:` line is the message's last `OBX` - with nothing after
the colon. Not a parsing failure: the source report itself never carries a conclusion
for this patient, and that's worth keeping visible rather than papering over with a
placeholder.

## Step 2: which LOINC coding and/or panels

[nw-gmsa.github.io/en/HaematoOncologyPathway.html](https://nw-gmsa.github.io/en/HaematoOncologyPathway.html)'s
own proposal names several LOINC panel/result codes for exactly this - reused below
rather than picked independently. Worth being upfront that this is genuinely
exploratory: HL7's own [Genomics Reporting IG's cytogenomics
page](https://build.fhir.org/ig/HL7/genomics-reporting/cytogenomics.html) is a stub
("The Clinical Genomics committee is reviewing this use case. It has not yet been
prioritized") - there's no published, confirmed cytogenomics `Observation` profile to
build against the way `05-test-results-from-vcf.ipynb`/`06-eu-laboratory-report-fhir-document.ipynb`
could check variant `Observation`s against real genomics-reporting IG examples. NW-GMSA's
own LOINC choices are the best available grounding here, not a confirmed standard.

| LOINC code | Meaning | Used for |
| --- | --- | --- |
| `62356-1` | Chromosome analysis result in ISCN expression | The karyotype string itself |
| `62367-8` | Chromosome analysis panel by FISH | The FISH finding, as a small panel |
| `50684-0` | Chromosome analysis.interphase [Interpretation] in Blood by FISH Narrative | A more specific alternative to `62367-8`'s free text, not used below |
| `62389-2` | Chromosome analysis master panel | Groups the karyotype (and FISH, where present) `Observation`s together |
| `62361-1` | Cells counted [#] | The FISH `Observation`'s "interphase cells examined" `component` |
| `84922-4` / `84921-6` | Cells.chromosome 7 monosomy / Cells.chromosome region 7q31 deletion, per Cells counted in Blood or Tissue by FISH | LOINC's per-abnormality "cells positive" pattern - neither matches this report's actual `7q36.1` probe closely enough to use, see below |
| `33893-9` | Karyotype [Identifier] in Bone marrow Nominal | `DiagnosticReport.code` - already what the live RIE assigns today |

`62386-8` (summary panel), `77314-3` (basic associated observations panel), `62343-9`
(microarray panel), `62349-6` (fetal/prenatal G-banded panel), and `82255-1` (marker
and derivative chromosome document) are the rest of the same LOINC family the page
names - not used here since neither fixture involves a microarray, a prenatal
referral, or a standalone marker-chromosome document, but the same pattern below would
apply to them.

In [6]:
KARYOTYPE_ISCN_CODE = {"system": LOINC_SYSTEM, "code": "62356-1", "display": "Chromosome analysis result in ISCN expression"}
FISH_PANEL_CODE = {"system": LOINC_SYSTEM, "code": "62367-8", "display": "Chromosome analysis panel by FISH"}
MASTER_PANEL_CODE = {"system": LOINC_SYSTEM, "code": "62389-2", "display": "Chromosome analysis master panel"}
DIAGNOSTIC_REPORT_CODE = {"system": LOINC_SYSTEM, "code": "33893-9", "display": "Karyotype [Identifier] in Bone marrow Nominal"}

### Building the karyotype `Observation` (`62356-1`)

`Observation.valueString` carries the ISCN expression itself - still a string, since
ISCN notation isn't further decomposable into FHIR's own value types without a
purpose-built parser, but now a single, queryable, correctly-escaped value instead of
one line lost inside a page of prose. `Observation.note` keeps the plain-language
paragraph explaining what the karyotype means, for a human reader.

In [7]:
def plain_language_paragraph(text, start_marker):
    lines = text.split("\n")
    start = next(i for i, l in enumerate(lines) if l.strip().startswith(start_marker)) + 1
    while lines[start].strip() == "":
        start += 1
    paragraph = []
    for l in lines[start:]:
        if l.strip() == "" and paragraph:
            break
        if l.strip():
            paragraph.append(l.strip())
    return " ".join(paragraph)


def karyotype_observation(patient_fullurl, karyotype, plain_language):
    return {
        "resourceType": "Observation",
        "status": "final",
        "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "laboratory"}]}],
        "code": {"coding": [KARYOTYPE_ISCN_CODE]},
        "subject": {"reference": patient_fullurl},
        "valueString": karyotype,
        "note": [{"text": plain_language}],
    }


shire1_karyotype, shire1_amendment = extract_karyotype(shire1_text)
shire1_plain = plain_language_paragraph(shire1_text, "Karyotype:")

shire2_karyotype, shire2_amendment = extract_karyotype(shire2_text)
shire2_plain = plain_language_paragraph(shire2_text, "Chromosome analysis has shown")

print(json.dumps(karyotype_observation("urn:uuid:patient", shire1_karyotype, shire1_plain), indent=2))

{
  "resourceType": "Observation",
  "status": "final",
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/observation-category",
          "code": "laboratory"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "http://loinc.org",
        "code": "62356-1",
        "display": "Chromosome analysis result in ISCN expression"
      }
    ]
  },
  "subject": {
    "reference": "urn:uuid:patient"
  },
  "valueString": "46,XY,del(20)(q*q*)[]",
  "note": [
    {
      "text": "Chromosome analysis has shown an abnormal karyotype with a deletion in the long arm of chromosome 20."
    }
  ]
}


### Building the FISH `Observation` (`62367-8`) - `Shire-2` only

`Shire-1` has no FISH result - only `Shire-2` gets this `Observation`. The interphase
cell count (*"93 out of 100 interphase cells..."*) is exactly the kind of number the
pathway page calls out as deserving to be "a discrete quantity rather than narrative
prose" - two `component`s, and this time one of them *is* codeable:

- **"Interphase cells examined"** → LOINC `62361-1` "Cells counted [#]" - a real,
  generic count code, confirmed.
- **"Interphase cells positive"** → LOINC does model this, but per-abnormality rather
  than generically - e.g. `84922-4` "Cells.chromosome 7 monosomy/Cells counted in
  Blood or Tissue by FISH", `84921-6` "Cells.chromosome region 7q31 deletion/Cells
  counted in Blood or Tissue by FISH". Neither is quite right here, though: this
  report's probe targets `7q36.1` specifically (*"loss of the 7q36.1 gene probe"*),
  not `7q31` or plain monosomy 7 - close, but a different locus, so using either would
  misrepresent which probe was actually read. Left `code.text`-only rather than
  reaching for the nearest-sounding code - the same honesty-over-guessing convention
  this repo's other notebooks follow for genuinely uncoded fields, and a concrete
  example of *why* LOINC's per-abnormality pattern needs the exact code confirmed
  against the lab's actual probe, not assumed from a similar-looking one.

In [8]:
CELLS_COUNTED_CODE = {"system": LOINC_SYSTEM, "code": "62361-1", "display": "Cells counted [#]"}


def fish_observation(patient_fullurl, fish):
    return {
        "resourceType": "Observation",
        "status": "final",
        "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "laboratory"}]}],
        "code": {"coding": [FISH_PANEL_CODE]},
        "subject": {"reference": patient_fullurl},
        "valueString": fish["result"],
        "component": [
            {"code": {"text": "Interphase cells positive"}, "valueQuantity": {"value": fish["positive_cells"], "unit": "cells"}},
            {"code": {"coding": [CELLS_COUNTED_CODE]}, "valueQuantity": {"value": fish["cells_examined"], "unit": "cells"}},
        ],
    }


shire2_fish = extract_fish(shire2_text)
print(json.dumps(fish_observation("urn:uuid:patient", shire2_fish), indent=2))

{
  "resourceType": "Observation",
  "status": "final",
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/observation-category",
          "code": "laboratory"
        }
      ]
    }
  ],
  "code": {
    "coding": [
      {
        "system": "http://loinc.org",
        "code": "62367-8",
        "display": "Chromosome analysis panel by FISH"
      }
    ]
  },
  "subject": {
    "reference": "urn:uuid:patient"
  },
  "valueString": "7q deletion detected.",
  "component": [
    {
      "code": {
        "text": "Interphase cells positive"
      },
      "valueQuantity": {
        "value": 93,
        "unit": "cells"
      }
    },
    {
      "code": {
        "coding": [
          {
            "system": "http://loinc.org",
            "code": "62361-1",
            "display": "Cells counted [#]"
          }
        ]
      },
      "valueQuantity": {
        "value": 100,
        "unit": "cells"
      }
    }
  ]
}


### Grouping them: the master panel (`62389-2`)

The standard FHIR panel pattern - an `Observation` with no value of its own,
`hasMember` pointing at the actual findings. `Shire-1`'s panel has one member
(karyotype only); `Shire-2`'s has two (karyotype and FISH).

In [9]:
def master_panel_observation(patient_fullurl, member_fullurls):
    return {
        "resourceType": "Observation",
        "status": "final",
        "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/observation-category", "code": "laboratory"}]}],
        "code": {"coding": [MASTER_PANEL_CODE]},
        "subject": {"reference": patient_fullurl},
        "hasMember": [{"reference": ref} for ref in member_fullurls],
    }

### `DiagnosticReport`: conclusion, and amendments instead of inline text

`DiagnosticReport.code` reuses `33893-9` - already what the live RIE assigns, so
nothing changes there. `conclusion` is the extracted plain-text summary (empty for
`Shire-1`, as found above). `conclusionCode` is deliberately left out - the pathway
page itself says the coded diagnosis/impression "remain[s] to be confirmed with the
laboratories", so this notebook doesn't invent one. Where an amendment marker was
found, `status` becomes `corrected` rather than `final`.

In [10]:
def diagnostic_report(patient_fullurl, panel_fullurl, conclusion, amendment):
    report = {
        "resourceType": "DiagnosticReport",
        "status": "corrected" if amendment else "final",
        "category": [{"coding": [{"system": "http://terminology.hl7.org/CodeSystem/v2-0074", "code": "GE"}]}],
        "code": {"coding": [DIAGNOSTIC_REPORT_CODE]},
        "subject": {"reference": patient_fullurl},
        "result": [{"reference": panel_fullurl}],
    }
    if conclusion:
        report["conclusion"] = conclusion
    return report

### Assembling both as FHIR Message `R01` `Bundle`s

`Bundle.type = "message"`, led by a `MessageHeader` - the same shape every other `R01`
in this repo uses, not the plain `"collection"` a bare resource list would be. Both
`MessageHeader` *and* `Patient` are reused verbatim from the live RIE's own
already-committed output (`Output/FHIR/R01/Shire-1.txt.json`/`Shire-2.txt.json`) -
`Bundle.identifier`/`.timestamp` too, straight off `MSH-10`/`MSH-7` via that earlier
real `transformToFHIR` run - only `MessageHeader.focus` is repointed, from the old
single free-text `Observation` to this notebook's own `DiagnosticReport`.

In [11]:
def build_bundle(v2_path, existing_fhir_path, output_path):
    text, _ = report_text(v2_path)
    with open(existing_fhir_path) as f:
        existing_bundle = json.load(f)
    patient = next(e["resource"] for e in existing_bundle["entry"] if e["resource"]["resourceType"] == "Patient")
    message_header = json.loads(json.dumps(
        next(e["resource"] for e in existing_bundle["entry"] if e["resource"]["resourceType"] == "MessageHeader")
    ))
    patient_fullurl = f"urn:uuid:{uuid4()}"

    karyotype, amendment = extract_karyotype(text)
    plain = plain_language_paragraph(text, "Karyotype:")
    conclusion = extract_conclusion(text)
    fish = extract_fish(text)

    karyotype_fullurl = f"urn:uuid:{uuid4()}"
    entries = [{"fullUrl": patient_fullurl, "resource": patient},
               {"fullUrl": karyotype_fullurl, "resource": karyotype_observation(patient_fullurl, karyotype, plain)}]
    member_fullurls = [karyotype_fullurl]

    if fish and fish["result"]:
        fish_fullurl = f"urn:uuid:{uuid4()}"
        entries.append({"fullUrl": fish_fullurl, "resource": fish_observation(patient_fullurl, fish)})
        member_fullurls.append(fish_fullurl)

    panel_fullurl = f"urn:uuid:{uuid4()}"
    entries.append({"fullUrl": panel_fullurl, "resource": master_panel_observation(patient_fullurl, member_fullurls)})

    report_fullurl = f"urn:uuid:{uuid4()}"
    entries.append({"fullUrl": report_fullurl, "resource": diagnostic_report(patient_fullurl, panel_fullurl, conclusion, amendment)})

    message_header["focus"] = [{"reference": report_fullurl}]
    entries.insert(0, {"fullUrl": f"urn:uuid:{uuid4()}", "resource": message_header})

    bundle = {
        "resourceType": "Bundle",
        "identifier": existing_bundle["identifier"],
        "timestamp": existing_bundle["timestamp"],
        "type": "message",
        "entry": entries,
    }
    with open(output_path, "w") as f:
        json.dump(bundle, f, indent=2)
    return bundle


shire1_bundle = build_bundle("Input/V2/R01/Shire-1.txt", "Output/FHIR/R01/Shire-1.txt.json", "Output/FHIR/R01/Shire-1-structured.json")
shire2_bundle = build_bundle("Input/V2/R01/Shire-2.txt", "Output/FHIR/R01/Shire-2.txt.json", "Output/FHIR/R01/Shire-2-structured.json")

for label, bundle in (("Shire-1", shire1_bundle), ("Shire-2", shire2_bundle)):
    print(label, "->", [e["resource"]["resourceType"] for e in bundle["entry"]])

Shire-1 -> ['MessageHeader', 'Patient', 'Observation', 'Observation', 'DiagnosticReport']
Shire-2 -> ['MessageHeader', 'Patient', 'Observation', 'Observation', 'Observation', 'DiagnosticReport']


### The same LOINC codes, directly in HL7 v2

None of the above needed `V2_TOOLS`'s `/transformToFHIR` - the extraction in Step 1
already produced everything needed to add new, *coded* `OBX` segments straight onto
the original message, alongside the existing free-text ones, no FHIR involved at all.
Same three LOINC codes as the `Observation`s above, just carried as `OBX-3` (`CE`:
`<code>^<text>^<coding system>`, `LN` for LOINC per v2 table `0396`) instead of
`Observation.code`. Set IDs continue from the message's own last `OBX` (`11` for
`Shire-1`, `31` for `Shire-2`) rather than restarting.

In [12]:
def v2_field(value):
    """Escape a value for safe placement in a v2 field - the same four characters
    Step 1's decode_v2_escapes() reverses, done here in the opposite direction."""
    return (str(value).replace("\\", "\\E\\").replace("|", "\\F\\")
                       .replace("^", "\\S\\").replace("~", "\\R\\").replace("&", "\\T\\"))


def obx_segment(set_id, value_type, code, value, units=""):
    return f"OBX|{set_id}|{value_type}|{code}||{v2_field(value)}|{units}||||||F"


def structured_obx_segments(text, start_set_id):
    karyotype, amendment = extract_karyotype(text)
    fish = extract_fish(text)

    karyotype_code = f"{KARYOTYPE_ISCN_CODE['code']}^{KARYOTYPE_ISCN_CODE['display']}^LN"
    segments = [obx_segment(start_set_id, "ST", karyotype_code, karyotype)]
    set_id = start_set_id + 1

    if fish and fish["result"]:
        fish_code = f"{FISH_PANEL_CODE['code']}^{FISH_PANEL_CODE['display']}^LN"
        segments.append(obx_segment(set_id, "ST", fish_code, fish["result"]))
        set_id += 1

        segments.append(obx_segment(set_id, "NM", "^Interphase cells positive", fish["positive_cells"]))
        set_id += 1

        cells_counted_code = f"{CELLS_COUNTED_CODE['code']}^{CELLS_COUNTED_CODE['display']}^LN"
        segments.append(obx_segment(set_id, "NM", cells_counted_code, fish["cells_examined"], units="{cells}^{cells}^UCUM"))
        set_id += 1

    return segments


def add_structured_obx(v2_path, output_path):
    segments = read_segments(v2_path)
    text, by_seg = report_text(v2_path)
    last_set_id = int(by_seg["OBX"][-1][1])
    new_obx = structured_obx_segments(text, last_set_id + 1)
    with open(output_path, "w", newline="") as f:
        f.write("\r".join(segments + new_obx) + "\r")
    return new_obx


for label, v2_path in (("Shire-1", "Input/V2/R01/Shire-1.txt"), ("Shire-2", "Input/V2/R01/Shire-2.txt")):
    new_obx = add_structured_obx(v2_path, f"Output/V2/R01/{label}-structured.txt")
    print(f"=== {label}: OBX segments appended ===")
    for segment in new_obx:
        print(segment)
    print()

=== Shire-1: OBX segments appended ===
OBX|12|ST|62356-1^Chromosome analysis result in ISCN expression^LN||46,XY,del(20)(q*q*)[]|||||||F

=== Shire-2: OBX segments appended ===
OBX|32|ST|62356-1^Chromosome analysis result in ISCN expression^LN||48\R\50,XY,+X,-Y,-3,-6,-7,del(7)(q?31q36),+8,+11,add(12)(p11.2),+14,+15,-16,+19,+20,+22-,+1\R\2mar[cp10]|||||||F
OBX|33|ST|62367-8^Chromosome analysis panel by FISH^LN||7q deletion detected.|||||||F
OBX|34|NM|^Interphase cells positive||93|||||||F
OBX|35|NM|62361-1^Cells counted [#]^LN||100|{cells}^{cells}^UCUM||||||F



### A basic structural check

No NW-GMSA IG profile exists for any of this yet (see Step 2's caveat), so there's
nothing to validate *against* - but the FHIR R4 base spec itself is still worth
checking: well-formed JSON, correct cardinalities, valid code systems. Base-spec
validation only, not IG conformance.

In [13]:
def validate_base_r4(path):
    outcome_path = "Results/FHIR/R01/" + os.path.basename(path) + "-OperationOutcome.json"
    subprocess.run(
        ["java", "-jar", "validator_cli.jar", path, "-version", "4.0.1",
         "-tx", "n/a", "-output", outcome_path, "-output-style", "json"],
        capture_output=True,
    )
    with open(outcome_path) as f:
        outcome = json.load(f)
    errors = [i for i in outcome["issue"] if i["severity"] in ("error", "fatal")]
    return errors


for path in ("Output/FHIR/R01/Shire-1-structured.json", "Output/FHIR/R01/Shire-2-structured.json"):
    errors = validate_base_r4(path)
    print(path, "->", f"{len(errors)} error(s)" if errors else "no structural errors")
    for e in errors:
        print("  ", e.get("details", {}).get("text", e))

Output/FHIR/R01/Shire-1-structured.json -> no structural errors


Output/FHIR/R01/Shire-2-structured.json -> no structural errors


No `error`/`fatal`-severity issues either run - the remaining `warning`/
`information` noise (missing `Resource.text` narrative, `Observation.performer`/
`.effective[x]`, an `IdentifierType` binding note on the *copied* `Patient` - see
"Assembling both bundles" above) is base-spec best-practice guidance, not something
this notebook's own construction got wrong. Package resolution for
`fhir.nwgenomics.nhs.uk` happens over the network on each run (see the loader's own
`Failed to determine latest version... using that [local copy]` messages if you run
this yourself), so which exact `warning`s surface can vary between runs - the absence
of any `error`/`fatal` is the part worth relying on.

## Summary

- Today's real output for both files is one `Observation` (`valueString` = the whole
  report as one string) plus a `DiagnosticReport` whose only coded field is `code`
  itself (`33893-9`, LOINC).
- Extracting the individual variables out of that free text is mostly plain regex over
  the reassembled `OBX-5` text - with one real HL7 v2 wrinkle worth knowing about:
  `\R\` (and the other `\X\` escapes) need decoding before a wrapped multi-`OBX`
  value like an ISCN karyotype string is actually correct.
- NW-GMSA's own `HaematoOncologyPathway.html` names the LOINC codes to structure it
  with (`62356-1` for the karyotype, `62367-8` for FISH, `62389-2` to group them as a
  panel) - genuinely exploratory, since HL7's own Genomics Reporting IG doesn't yet
  have confirmed cytogenomics guidance to check against.
- `DiagnosticReport.conclusion` carries the free-text summary (empty where the source
  message's own `Conclusion:` line is empty - not invented), `conclusionCode` is left
  out entirely (not yet confirmed, per the page itself), and `status = corrected`
  replaces the inline `-Amendment <date>` markers with a formal status change (the
  page's own proposal also links a `Provenance` history to record *when* - not built
  here, kept to what's directly demonstrable from the source message's own status).
- The same three LOINC codes carry straight onto new, coded `OBX` segments appended to
  the original v2 message too - no `/transformToFHIR` involved, just the values Step 1
  already extracted, `CE`-coded (`<code>^<text>^LN`) instead of `Observation.code`.
- None of this needed a live `V2_TOOLS`/`V2_SERVER` connection - every cell above runs
  against the local fixtures and the real engine's own already-committed output.